# 01 — Inspeção dos dados brutos (Fase 1)

**Projeto Z.Data — Insper Data, Ciclo Básico 2026.2**

Pergunta do projeto: indicadores contábeis estão associados à **entrada** de companhias abertas brasileiras não financeiras em *distress* financeiro nos dois anos seguintes?

Este notebook **só olha** os dados: não cria variáveis nem filtra nada. O objetivo é saber exatamente o que veio do Economatica antes de qualquer transformação, para que cada decisão da Fase 2 tenha uma justificativa verificável.

Regras que valem para o projeto inteiro:
- `data/raw/` nunca é editado; tudo é refeito por código a partir dele.
- `"-"` e células vazias do Economatica viram **NaN, nunca zero**. Zero seria uma informação falsa ("a empresa não tem dívida"), enquanto NaN diz a verdade ("não sabemos").
- Constantes e escolhas de projeto ficam em `src/config.py`; funções reutilizáveis ficam em `src/`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config as cfg, dados, verificacoes as vf, style

style.aplicar()
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: style.num_br(x, 2))
print('Fonte dos gráficos:', style.FONTE)

Fonte dos gráficos: Montserrat


## 1. O que há em `data/raw/`

Duas fontes:
- **Economatica**: três exportações (o Economatica limita o número de colunas por exportação, por isso vieram separadas).
- **CVM (Dados Abertos)**: o cadastro de companhias abertas (`cad_cia_aberta.csv`) e os Formulários Cadastrais (FCA) de 2010 a 2026, que trazem o **código de negociação** de cada companhia. O FCA é necessário porque a base do Economatica identifica as empresas só pelo ticker, sem CNPJ.

In [2]:
arquivos = sorted(p for p in cfg.DATA_RAW.rglob('*') if p.is_file())
pd.DataFrame({
    'arquivo': [str(p.relative_to(cfg.DATA_RAW)) for p in arquivos],
    'tamanho (KB)': [round(p.stat().st_size / 1024) for p in arquivos],
})

,arquivo,tamanho (KB)
0,cvm\cad_cia_aberta.csv,1459
1,cvm\fca\fca_2010.zip,330
2,cvm\fca\fca_2011.zip,372
3,cvm\fca\fca_2012.zip,372
4,cvm\fca\fca_2013.zip,364
5,cvm\fca\fca_2014.zip,376
6,cvm\fca\fca_2015.zip,372
7,cvm\fca\fca_2016.zip,367
8,cvm\fca\fca_2017.zip,366
9,cvm\fca\fca_2018.zip,373


### 1.1 As três exportações do Economatica

In [3]:
brutos = {f.name: dados.ler_economatica_bruto(f) for f in cfg.ARQ_ECONOMATICA}
pd.DataFrame([vf.resumo_arquivo(b, n) for n, b in brutos.items()]).set_index('arquivo')

,linhas,colunas,empresas (Ativo),períodos (Data),primeiro período,último período,duplicatas Ativo+Data
arquivo,,,,,,,
Economatica-8900590473-20261020215038.csv,19856,20,292,68,1T2010,4T2026,0
extra.csv,19294,5,295,67,1T2010,3T2025,0
juros_pagaos_&_capex.csv,19294,4,295,67,1T2010,3T2025,0


In [4]:
# Colunas: nome original, nome curto usado no projeto, tipo e % de vazios
linhas = []
for n, b in brutos.items():
    for c in b.columns:
        linhas.append({'arquivo': n, 'coluna original': c, 'nome no projeto': dados._nome_curto(c),
                       'tipo': str(b[c].dtype), '% NaN': b[c].isna().mean()})
colunas = pd.DataFrame(linhas)
colunas.style.format({'% NaN': lambda x: style.pct_br(x, 1)}).hide(axis='index')

arquivo,coluna original,nome no projeto,tipo,% NaN
Economatica-8900590473-20261020215038.csv,Ativo,Ativo,str,"0,0%"
Economatica-8900590473-20261020215038.csv,Data,Data,str,"0,0%"
Economatica-8900590473-20261020215038.csv,DesFin| Em moeda orig| em milhares| no exercício| consolid:sim*,desp_financeira,float64,"27,4%"
Economatica-8900590473-20261020215038.csv,Ebitda/Fin| no exercício| consolid:sim*,ebitda_desp_fin,float64,"30,7%"
Economatica-8900590473-20261020215038.csv,Ativo Tot| Em moeda orig| em milhares| consolid:sim*,ativo_total,float64,"25,3%"
Economatica-8900590473-20261020215038.csv,Div Tt Bruta| Em moeda orig| em milhares| consolid:sim*,divida_bruta,float64,"25,4%"
Economatica-8900590473-20261020215038.csv,Div Tt Lq| Em moeda orig| em milhares| consolid:sim*,divida_liquida,float64,"25,4%"
Economatica-8900590473-20261020215038.csv,Patrim Liq| Em moeda orig| em milhares| consolid:sim*,patrimonio_liquido,float64,"25,3%"
Economatica-8900590473-20261020215038.csv,LiqCor| consolid:sim*,liquidez_corrente,float64,"25,3%"
Economatica-8900590473-20261020215038.csv,CaixaEEqCx| Em moeda orig| em milhares| consolid:sim*,caixa,float64,"27,9%"


**Formato.** As três exportações estão em formato **longo**: uma linha por empresa (`Ativo`, ticker com o sufixo da bolsa, ex.: `PETR4<XBSP>`) × trimestre (`Data`, ex.: `1T2010`), de 1T2010 a 4T2026, com um indicador por coluna. Os nomes das colunas trazem a definição do Economatica: moeda, unidade ("em milhares"), janela ("de 12 meses" ou "no exercício") e consolidação.

Os pontos que importam para a análise:
- **Janelas diferentes.** `DesFin`, `Ebitda/Fin` e `Receita` estão "no exercício", ou seja, acumuladas no ano. `EBIT`, `EBITDA`, `Lucro Liquido` e `CxOper` estão "de 12 meses". Na seção 3a testamos isso nos dados.
- **Unidades diferentes.** `CxOper` e `JurosPagos` não têm "em milhares" no nome: estão em R$. Na carga dividimos as duas por 1.000 para ficar tudo em milhares de R$ (a conferência está na seção 3a).
- **`ResLuc` é Reservas de Lucro**, uma conta do patrimônio líquido, e não "resultado líquido". Na seção 3c mostramos isso nos dados.
- `JurosPagos` e `Capex` vieram 100% vazios e não serão usados, como combinado.

### 1.2 Base trimestral única

Juntamos as três exportações por `Ativo` + `Data` com *outer join*, para não perder nenhuma linha. A função está em `src/dados.py` e também padroniza os nomes e as unidades.

In [5]:
df = dados.carregar_economatica()
print('linhas:', len(df), '| colunas:', df.shape[1])
print('tickers:', df['ticker'].nunique(), '| trimestres:', df['Data'].nunique())

tk = {n: set(b['Ativo'].str.replace(r'<.*>', '', regex=True)) for n, b in brutos.items()}
nomes = list(tk)
print('\nTickers só nas exportações extra/juros (não na principal):', sorted(tk[nomes[1]] - tk[nomes[0]]))
print('Tickers só na principal:', sorted(tk[nomes[0]] - tk[nomes[1]]))

linhas: 20044 | colunas: 27
tickers: 295 | trimestres: 68

Tickers só nas exportações extra/juros (não na principal): ['B3SA3', 'CSUD3', 'G2DI33']
Tickers só na principal: []


In [6]:
# Quantas empresas têm balanço (ativo total) em cada trimestre
disp = df[df['ativo_total'].notna()].pivot_table(index='ano', columns='trimestre', values='ticker', aggfunc='count')
sem_dado = df.groupby('ticker')['ativo_total'].apply(lambda s: s.notna().sum() == 0)
print('Tickers sem nenhum balanço no período inteiro:', sem_dado.sum(), sorted(sem_dado[sem_dado].index))
disp

Tickers sem nenhum balanço no período inteiro: 3 ['B3SA3', 'CSUD3', 'G2DI33']


trimestre,1,2,3,4
ano,,,,
2010,"167,00","167,00","167,00","174,00"
2011,"174,00","175,00","175,00","177,00"
2012,"176,00","177,00","177,00","182,00"
2013,"183,00","183,00","183,00","184,00"
2014,"184,00","185,00","185,00","192,00"
2015,"191,00","193,00","193,00","200,00"
2016,"196,00","195,00","195,00","211,00"
2017,"202,00","201,00","201,00","237,00"
2018,"206,00","206,00","207,00","249,00"


**Leitura.** O painel **cresce** de cerca de 170 empresas com balanço em 2010 para cerca de 280 em 2024–2025. Isso é esperado se o *Screening* trouxe só as empresas listadas hoje: as que abriram capital depois (a onda de IPOs de 2020–21) entram no meio do período, e as que fecharam capital antes de 2026 não aparecem. O 3T2026 e o 4T2026 ainda não foram divulgados. Esse padrão é a primeira evidência do **viés de sobrevivência**, que medimos na seção 3g e de novo na Fase 3.

## 2. `"-"` vira NaN, nunca zero

O Economatica usa `"-"` para "sem dado". Na leitura (`na_values=['-', '']`) esses valores viram NaN. A tabela abaixo confere que nenhuma coluna numérica ficou como texto e mostra quantos zeros **verdadeiros** existem, ou seja, os que vieram como `0` no arquivo e não como `"-"`.

In [7]:
num = df.select_dtypes('number').columns.drop(['ano', 'trimestre'])
print('Colunas que não viraram número:', [c for c in df.columns if c not in num and c not in ('ticker', 'Data', 'ano', 'trimestre')])
pd.DataFrame({'% NaN': df[num].isna().mean(), 'zeros': (df[num] == 0).sum()}).style.format({'% NaN': lambda x: style.pct_br(x, 1)})

Colunas que não viraram número: []


,% NaN,zeros
desp_financeira,"28,0%",48
ebitda_desp_fin,"31,3%",1
ativo_total,"26,0%",0
divida_bruta,"26,1%",514
divida_liquida,"26,1%",7
patrimonio_liquido,"26,0%",0
liquidez_corrente,"26,0%",0
caixa,"28,6%",47
emprestimos_cp,"26,1%",659
emprestimos_lp,"26,1%",943


## 3. Verificações obrigatórias

### 3a. 12 meses × "no exercício"

Se uma variável é **acumulada no ano**, o valor sobe ao longo do ano (o 2T é cerca de 2× o 1T e o 4T é cerca de 4×) e despenca no 1T seguinte, quando o acumulado zera. Se é **de 12 meses**, os quatro trimestres têm valores parecidos (razão em torno de 1). O teste usa só empresa-anos com os quatro trimestres positivos.

In [8]:
acum = vf.padrao_acumulado(df, ['ebit', 'receita', 'desp_financeira', 'lucro_liquido_12m', 'caixa_operacional_12m', 'ebitda'])
acum.style.format({c: lambda x: style.num_br(x, 2) for c in ['T2/T1', 'T3/T1', 'T4/T1']}
                  | {c: lambda x: style.pct_br(x, 1) for c in ['% T1<T2<T3<T4', '% T1 seguinte < T4']})

,empresa-anos,T2/T1,T3/T1,T4/T1,% T1<T2<T3<T4,% T1 seguinte < T4,parece acumulada
variável,,,,,,,
ebit,2563,"1,03","1,07","1,10","30,3%","40,1%",False
receita,3445,"2,05","3,17","4,34","96,7%","97,8%",True
desp_financeira,3285,"2,02","3,10","4,25","92,4%","96,1%",True
lucro_liquido_12m,2100,"1,03","1,06","1,07","24,7%","43,2%",False
caixa_operacional_12m,2290,"1,03","1,07","1,09","19,0%","47,3%",False
ebitda,2611,"1,03","1,06","1,09","33,2%","37,1%",False


In [9]:
# Conferência de unidade do caixa operacional (exportado em R$, convertido para milhares na carga):
# depois da conversão, o FCO deve ter a mesma ordem de grandeza do EBITDA (ambos de 12 meses, em milhares).
q4 = df[df['trimestre'] == 4]
razao = (q4['caixa_operacional_12m'] / q4['ebitda']).replace([np.inf, -np.inf], np.nan).dropna()
print('Mediana FCO / EBITDA (após dividir FCO por 1.000):', style.num_br(razao.median(), 2))
df.loc[(df['ticker'] == 'PETR4') & (df['trimestre'] == 4) & df['ano'].between(2010, 2013),
       ['ticker', 'Data', 'ebitda', 'caixa_operacional_12m', 'lucro_liquido_12m']]

Mediana FCO / EBITDA (após dividir FCO por 1.000): 0,61


,ticker,Data,ebitda,caixa_operacional_12m,lucro_liquido_12m
14267,PETR4,4T2010,"60.455.664,00","53.435.437,00","35.189.366,00"
14271,PETR4,4T2011,"61.968.216,00","56.322.101,00","33.313.097,00"
14275,PETR4,4T2012,"53.242.133,00","54.145.734,00","21.182.442,00"
14279,PETR4,4T2013,"62.823.684,00","56.209.345,00","23.570.367,00"


**Resultado.**
- **Acumuladas no ano:** `receita` e `desp_financeira` (e, por consequência, `ebitda_desp_fin`, que divide o EBITDA de 12 meses pela despesa financeira acumulada). O 4T é cerca de 4× o 1T, e o valor sobe quase sempre ao longo do ano e cai no 1T seguinte.
- **De 12 meses:** `ebit`, `ebitda`, `lucro_liquido_12m` e `caixa_operacional_12m` têm razões em torno de 1 entre os trimestres.
- **Por que isso não atrapalha:** na Fase 2 ficamos só com o **4º trimestre**. No 4T, "no exercício" é o ano completo e "12 meses" também é o ano completo, então EBIT e despesa financeira passam a se referir ao mesmo período. Se usássemos outro trimestre, a cobertura `ebit / desp_financeira` dividiria 12 meses de EBIT por 3, 6 ou 9 meses de despesa e ficaria inflada.
- **Unidade do FCO:** depois de dividir por 1.000, o FCO tem a mesma ordem de grandeza do EBITDA, o que confirma que ele estava em R$ e não em milhares. Exemplo: a Petrobras tem FCO de R$ 49 bilhões em 2009 e 2010, coerente com o EBITDA de cerca de R$ 60 bilhões.

### 3b. Sinal da despesa financeira

In [10]:
pd.DataFrame({'todos os trimestres': vf.sinal(df, 'desp_financeira'),
              'só 4º trimestre': vf.sinal(df, 'desp_financeira', df['trimestre'] == 4)})

,todos os trimestres,só 4º trimestre
> 0,14287,3580
= 0,48,9
< 0,88,18
NaN,5621,1403


In [11]:
neg4 = df[(df['trimestre'] == 4) & (df['desp_financeira'] <= 0)]
print('4º trimestre com despesa financeira <= 0:', len(neg4), 'obs. de', neg4['ticker'].nunique(), 'empresas')
neg4[['ticker', 'Data', 'desp_financeira', 'ebit', 'ebitda_desp_fin']]

4º trimestre com despesa financeira <= 0: 27 obs. de 19 empresas


,ticker,Data,desp_financeira,ebit,ebitda_desp_fin
855,AMAR3,4T2019,"-34.935,00","66.932,00","-11,30"
3025,BRBI11,4T2018,"0,00","63.617,00",NaN
3029,BRBI11,4T2019,"0,00","103.287,00",NaN
3033,BRBI11,4T2020,"0,00","133.175,00",NaN
4341,CMIN3,4T2024,"-62.039,00","4.967.308,00","-98,61"
5595,CXSE3,4T2015,"0,00","590.894,00",NaN
5951,DIRR3,4T2019,"-40.941,00","193.017,00","-5,59"
5955,DIRR3,4T2020,"-44.116,00","221.363,00","-5,96"
6679,EMBJ3,4T2014,"-89.340,00","1.303.774,00","-22,17"
6683,EMBJ3,4T2015,"-1.060.534,00","1.103.067,00","-2,05"


**Resultado.** A convenção do Economatica é **despesa positiva**: mais de 96% das observações são maiores que zero. Os casos com sinal trocado (despesa negativa, em geral porque a variação cambial positiva entrou na linha de despesa financeira) ou com despesa zero são poucos. Na Fase 2, como combinado, essas observações ficam com cobertura NaN ("não classificável") e não com uma cobertura negativa sem sentido.

### 3c. `resultado_liquido` × `lucro_liquido_12m`: não existe "resultado líquido" na base

A verificação pedida pressupõe uma coluna de resultado líquido. Essa coluna **não existe**: o campo que eu tinha chamado de `resultado_liquido` na junção anterior é `ResLuc`, que no Economatica significa **Reservas de Lucro**, uma conta do patrimônio líquido. As evidências estão abaixo. Uma reserva de lucro:
- quase só muda no 4º trimestre, porque a destinação do lucro é feita no fechamento do exercício;
- é quase sempre maior ou igual a zero;
- acompanha o tamanho do patrimônio líquido, e não o lucro do ano.

In [12]:
pd.DataFrame({
    'reservas_lucro': vf.mudancas_entre_trimestres(df, 'reservas_lucro'),
    'lucro_liquido_12m': vf.mudancas_entre_trimestres(df, 'lucro_liquido_12m'),
}).rename_axis('trimestre (% que mudou em relação ao anterior)').style.format(lambda x: style.pct_br(x, 1))

,reservas_lucro,lucro_liquido_12m
trimestre (% que mudou em relação ao anterior),,
1,"38,8%","100,0%"
2,"50,5%","100,0%"
3,"37,4%","100,0%"
4,"75,6%","100,0%"


In [13]:
print('Sinal de reservas_lucro (todos os trimestres):')
print(vf.sinal(df, 'reservas_lucro').to_string())
print('\nCorrelação de Spearman no 4º trimestre:')
q4[['reservas_lucro', 'lucro_liquido_12m', 'patrimonio_liquido']].corr(method='spearman')

Sinal de reservas_lucro (todos os trimestres):
> 0    10988
= 0     3484
< 0      353
NaN     5219

Correlação de Spearman no 4º trimestre:


,reservas_lucro,lucro_liquido_12m,patrimonio_liquido
reservas_lucro,"1,00","0,74","0,77"
lucro_liquido_12m,"0,74","1,00","0,63"
patrimonio_liquido,"0,77","0,63","1,00"


In [14]:
neg_res = df[df['reservas_lucro'] < 0]
print('Observações com reservas de lucro negativas:', len(neg_res), '| empresas:', neg_res['ticker'].nunique())
neg_res.groupby('ticker').agg(n=('Data', 'size'), primeiro=('Data', 'first'), ultimo=('Data', 'last'),
                              exemplo_reservas=('reservas_lucro', 'first'),
                              exemplo_pl=('patrimonio_liquido', 'first')).sort_values('n', ascending=False).head(15)

Observações com reservas de lucro negativas: 353 | empresas: 55


,n,primeiro,ultimo,exemplo_reservas,exemplo_pl
ticker,,,,,
NEXP3,47,4T2014,2T2026,"-23.311,00","520.488,00"
BDLL4,42,1T2016,2T2026,"-11.589,00","310.567,00"
IGSN3,29,4T2018,2T2026,"-60.609,00","637.663,00"
CVCB3,23,4T2020,2T2026,"-1.767,00","45.560,00"
ONCO3,19,4T2021,2T2026,"-20.621,00","1.854.009,00"
RAIL3,17,2T2021,2T2026,"-35.670,00","15.449.257,00"
PDGR3,17,1T2013,1T2025,"-698.549,00","4.943.488,00"
MNDL3,14,4T2012,3T2016,"-36,00","31.809,00"
LOGN3,12,2T2014,1T2017,"-43.659,00","583.431,00"


**Resultado.** As evidências confirmam que `ResLuc` é **Reservas de Lucro**: o valor quase só muda no 4T, a correlação com o PL é maior que com o lucro, e a maioria das observações é maior ou igual a zero. Consequências:
- A coluna foi renomeada para `reservas_lucro` em `src/config.py`. Ela **não é descartada**: é exatamente o X2 do Z''-score de Altman (reservas de lucro / ativo total).
- A verificação "resultado líquido acumulado × lucro de 12 meses" não se aplica. A única medida de lucro é `lucro_liquido_12m`, que é de 12 meses (seção 3a) e será usada no ROA.
- **Reservas de lucro negativas (353 obs., 55 empresas).** Pela Lei 6.404/76 não existe reserva de lucro negativa. O negativo vem do **plano de contas padronizado do Economatica**, que soma a linha de reservas de lucro com a de **prejuízos acumulados** e também pode refletir ajustes de avaliação patrimonial. O sinal negativo significa que os prejuízos acumulados superaram as reservas. **Decisão (dupla, 21/10/2026):** ler a variável como o **saldo líquido de resultados retidos** e manter o sinal. Um valor negativo indica destruição patrimonial por prejuízos recorrentes, que é exatamente o que o X2 do Z'' quer capturar.

### 3d. `passivo_mais_pl` = `ativo_total`?

In [15]:
par = df[['ativo_total', 'passivo_mais_pl']].dropna()
iguais = np.isclose(par['ativo_total'], par['passivo_mais_pl'], rtol=1e-9, atol=0.5)
print(f'Observações com os dois preenchidos: {len(par)} | iguais: {style.pct_br(iguais.mean(), 2)}')
print('Linhas com só um dos dois preenchido:', int((df['ativo_total'].isna() != df['passivo_mais_pl'].isna()).sum()))

Observações com os dois preenchidos: 14827 | iguais: 99,99%
Linhas com só um dos dois preenchido: 0


**Resultado.** As duas colunas são iguais em 99,99% das observações (14.825 de 14.827), como manda a identidade contábil Ativo = Passivo + PL. As 2 exceções (MSPA4 no 2T2021 e SYNE3 no 4T2022) diferem em R$ 1 mil, ou seja, são arredondamento. `passivo_mais_pl` é redundante e será **descartada** na Fase 2.

### 3e. Duplicatas empresa-período

Duplicatas literais (mesmo `Ativo` + `Data`) já foram contadas na tabela 1.1: **zero** nas três exportações. Também verificamos duplicatas "disfarçadas", quando a mesma companhia (mesmo CNPJ) aparece com dois tickers, como ON e PN. Para isso é preciso o CNPJ, que vem da ligação com a CVM na seção 3f.

### 3f. Ligação com a CVM e verificação de financeiras

A base do Economatica **não tem coluna de setor nem de CNPJ**, só o ticker. Por isso a verificação de financeiras é feita pelo setor da CVM (`SETOR_ATIV`), depois de ligar ticker → CNPJ pelo FCA (valores mobiliários). A ligação segue 3 etapas, e a forma como cada ticker casou fica registrada:
1. **exato:** o código de negociação aparece igual no FCA;
2. **raiz:** as mesmas 4 letras (ex.: `MRSA3B` → `MRSA3`), só quando a raiz leva a um único CNPJ;
3. **manual:** tabela em `src/config.py`, montada procurando o nome da empresa no cadastro. Os casos marcados como `manual_conferir` são palpites que a dupla precisa confirmar.

O FCA só passou a informar o código de negociação a partir de 2018. Empresas que fecharam capital antes disso não casariam pelo ticker, mas como elas praticamente não estão na base (seção 3g), isso não causa perda aqui.

In [16]:
cad = dados.carregar_cad_cvm()
cad1 = dados.cad_por_cnpj(cad)
print('Cadastro CVM:', len(cad), 'linhas,', cad['CNPJ_CIA'].nunique(), 'CNPJs (há CNPJs com mais de um registro)')

mapa = dados.mapa_ticker_cnpj(sorted(df['ticker'].unique()))
mapa = mapa.merge(cad1[['CNPJ_CIA', 'DENOM_SOCIAL', 'SIT', 'DT_REG_PRIMEIRO', 'DT_CANCEL', 'MOTIVO_CANCEL',
                        'SETOR_ATIV', 'financeira']], on='CNPJ_CIA', how='left')
mapa['financeira'] = mapa['financeira'].astype('boolean').fillna(False).astype(bool)
mapa['como_casou'].value_counts()

Cadastro CVM: 2678 linhas, 2531 CNPJs (há CNPJs com mais de um registro)


como_casou
exato              272
manual              13
raiz                 6
sem_match            3
manual_conferir      1
Name: count, dtype: int64

In [17]:
print('Sem CNPJ:')
display(mapa[mapa['como_casou'] == 'sem_match'][['ticker']].T)
print('Casados à mão, a conferir:')
display(mapa[mapa['como_casou'] == 'manual_conferir'][['ticker', 'CNPJ_CIA', 'DENOM_SOCIAL', 'SETOR_ATIV']])
print('CNPJ casado mas ausente do cadastro:', mapa['CNPJ_CIA'].notna().sum() - mapa['DENOM_SOCIAL'].notna().sum())

Sem CNPJ:


,22,120,162
ticker,AURA33,G2DI33,LLBI3


Casados à mão, a conferir:


,ticker,CNPJ_CIA,DENOM_SOCIAL,SETOR_ATIV
95,EGGY3,81.616.807/0001-55,GRANJA FARIA S.A.,Alimentos


CNPJ casado mas ausente do cadastro: 0


In [18]:
# 3e (continuação): mesma companhia com mais de um ticker?
dup = mapa.dropna(subset=['CNPJ_CIA']).groupby('CNPJ_CIA')['ticker'].apply(list)
print('CNPJs com mais de um ticker na base:', dup[dup.str.len() > 1].to_dict())

CNPJs com mais de um ticker na base: {}


In [19]:
# Financeiras pelo setor da CVM
fin = mapa[mapa['financeira']]
print('Empresas da base em setor financeiro segundo a CVM:', len(fin))
fin[['ticker', 'DENOM_SOCIAL', 'SETOR_ATIV', 'SIT']]

Empresas da base em setor financeiro segundo a CVM: 8


,ticker,DENOM_SOCIAL,SETOR_ATIV,SIT
30,B1003,B100 S.A.,Intermediação Financeira,ATIVO
31,B3SA3,"B3 S.A. - BRASIL, BOLSA, BALCÃO",Bolsas de Valores/Mercadorias e Futuros,ATIVO
44,BRBI11,BRBI BR PARTNERS S.A,Emp. Adm. Part. - Bancos,ATIVO
82,CXSE3,CAIXA SEGURIDADE PARTICIPAÇÕES S.A.,Emp. Adm. Part. - Seguradoras e Corretoras,ATIVO
135,HBTS5,COMPANHIIA HABITASUL DE PARTICIPAÇÕES,Emp. Adm. Part. - Crédito Imobiliário,ATIVO
146,ITSA4,ITAUSA S.A.,Emp. Adm. Part. - Intermediação Financeira,ATIVO
223,QUAL3,QUALICORP CONSULTORIA E CORRETORA DE SEGUROS S.A.,Seguradoras e Corretoras,ATIVO
292,WIZC3,WIZ CO PARTICIPAÇÕES E CORRETAGEM DE SEGUROS S.A.,Seguradoras e Corretoras,ATIVO


In [20]:
# Setores da CVM presentes na base
mapa['SETOR_ATIV'].value_counts(dropna=False).to_frame('empresas')

,empresas
SETOR_ATIV,
"Construção Civil, Mat. Constr. e Decoração",31
Comércio (Atacado e Varejo),28
Energia Elétrica,21
"Máquinas, Equipamentos, Veículos e Peças",18
Serviços Transporte e Logística,17
Têxtil e Vestuário,15
Metalurgia e Siderurgia,15
"Agricultura (Açúcar, Álcool e Cana)",10
Alimentos,9


**Resultado.**
- **Ligação ticker → CNPJ:** 292 dos 295 tickers casaram (272 exatos, 6 pela raiz e 14 à mão). COMR3, IGSN3 e JBSS32 foram confirmadas pela dupla; EGGY3 (`manual_conferir`) ainda será conferida. Ficaram sem CNPJ AURA33 e G2DI33, que são BDRs de empresas estrangeiras e não constam do cadastro de companhias abertas brasileiras, e LLBI3 (em recuperação judicial, registro cancelado de ofício segundo a dupla), que não foi encontrada no cadastro. O caso da JBS é especial: desde 2025 o ticker JBSS32 é o BDR da JBS N.V. (holandesa), mas o histórico é o da JBS S.A., por isso foi ligado ao CNPJ brasileiro.
- **Duplicatas disfarçadas:** nenhuma companhia aparece com dois tickers.
- **Financeiras:** **sobraram 8 empresas** classificadas pela CVM em setores financeiros: B1003 (B100), B3SA3 (B3, a bolsa), BRBI11 (BR Partners, banco de investimento), CXSE3 (Caixa Seguridade), HBTS5 (Habitasul, crédito imobiliário), ITSA4 (Itaúsa, holding do Itaú), QUAL3 (Qualicorp) e WIZC3 (Wiz). **Nenhuma foi excluída**: a decisão fica para a dupla (ver o resumo). Qualicorp e Wiz estão como "corretoras de seguros" na CVM, mas são prestadoras de serviço e não carregam risco de balanço de seguradora. A B3SA3 não tem nenhum balanço na base.
- **Setor:** o `SETOR_ATIV` da CVM é granular (47 categorias para 292 empresas) e separa holdings ("Emp. Adm. Part. - Energia Elétrica") das operacionais do mesmo setor. Para usar como dummy, será preciso agrupar.

### 3g. Canceladas e empresas que pararam de reportar

Se o *Screening* tivesse trazido também as empresas que fecharam capital, veríamos muitas empresas com o último balanço antes de 2025. Pelo cadastro da CVM, centenas de companhias tiveram o registro cancelado desde 2010.

In [21]:
pu = vf.primeiro_ultimo_balanco(df)
print('Empresas com algum balanço:', len(pu))
print('Último balanço antes de 2025:', (pu['ultimo_ano'] < 2025).sum())
print('Primeiro balanço depois de 2010:', (pu['primeiro_ano'] > 2010).sum())
pu[pu['ultimo_ano'] < 2025].join(mapa.set_index('ticker')[['DENOM_SOCIAL', 'SIT', 'DT_CANCEL', 'MOTIVO_CANCEL']])

Empresas com algum balanço: 292
Último balanço antes de 2025: 6
Primeiro balanço depois de 2010: 118


,primeiro_ano,ultimo_ano,n_trimestres,DENOM_SOCIAL,SIT,DT_CANCEL,MOTIVO_CANCEL
ticker,,,,,,,
ASAI3,2019,2021,8,SENDAS DISTRIBUIDORA S.A.,ATIVO,NaT,NaN
BMKS3,2010,2017,32,BICICLETAS MONARK SA,ATIVO,NaT,NaN
CRPG5,2010,2017,29,TRONOX PIGMENTOS DO BRASIL S.A.,ATIVO,NaT,NaN
CTSA3,2010,2024,60,CIA TECIDOS SANTANENSE - EM RECUPERAÇÃO JUDICIAL,SUSPENSO(A) - DECISÃO ADM,NaT,NaN
GEPA4,2015,2023,34,RIO PARANAPANEMA ENERGIA SA,ATIVO,NaT,NaN
TELB4,2014,2017,13,TELEC. BRASILEIRAS S.A. - TELEBRÁS,ATIVO,NaT,NaN


In [22]:
# Situação na CVM das empresas da base
print(mapa['SIT'].value_counts(dropna=False).to_string())
canc_cvm = cad1[(cad1['SIT'] == 'CANCELADA') & (cad1['DT_CANCEL'].dt.year.between(cfg.ANO_INI, 2026)) & ~cad1['financeira']]
print('\nCompanhias não financeiras canceladas na CVM entre 2010 e 2026:', len(canc_cvm))
print('Dessas, presentes na base:', canc_cvm['CNPJ_CIA'].isin(mapa['CNPJ_CIA']).sum())

SIT
ATIVO                        290
NaN                            3
SUSPENSO(A) - DECISÃO ADM      2

Companhias não financeiras canceladas na CVM entre 2010 e 2026: 469
Dessas, presentes na base: 0


In [23]:
# Entradas por ano (primeiro balanço na base)
pu['primeiro_ano'].value_counts().sort_index().to_frame('empresas que entram').T

primeiro_ano,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
empresas que entram,174,3,5,2,8,9,11,27,15,10,4,8,4,5,4,2,1


**Resultado: o viés de sobrevivência é forte.**
- Das **469 companhias não financeiras canceladas na CVM entre 2010 e 2026, nenhuma está na base.** 290 das 295 empresas da base estão ativas hoje na CVM, e as outras 2 com CNPJ estão "suspensas" por decisão administrativa (CTSA3, Santanense, em recuperação judicial, e RSID3, Rossi).
- Só **6 empresas** têm o último balanço antes de 2025. Nenhuma delas cancelou o registro: são casos em que o Economatica parou de trazer dados de uma empresa ainda ativa (BMKS3, CRPG5 e TELB4 em 2017, ASAI3 em 2021, GEPA4 em 2023, CTSA3 em 2024). Esses casos precisam ser olhados na Fase 2, porque viram Y = NaN.
- **118 empresas** têm o primeiro balanço depois de 2010. Ou seja, o painel começa com 174 empresas e vai recebendo as que abriram capital depois.
- **Demonstrações anteriores ao IPO:** há 188 empresa-anos com **só o 4º trimestre** (concentrados entre 2016 e 2020). São as demonstrações anuais que a empresa publica no prospecto antes de abrir capital. Nesses anos ela ainda não era companhia aberta. Isso explica os saltos de 3T para 4T na tabela da seção 1.2 (por exemplo, 201 para 237 em 2017).

Conclusão: o *Screening* trouxe **só as empresas listadas hoje**. Os casos clássicos de distress que terminaram em cancelamento, falência ou fechamento de capital entre 2010 e 2026 estão fora. A taxa de eventos vai ser subestimada, principalmente nos primeiros anos. Isso será medido ano a ano na Fase 3 (base × CVM).

## 4. Outros achados da inspeção

In [24]:
# Dívida em moeda estrangeira: em quais anos tem dado?
print('divida_moeda_estrangeira não nula por ano:', df.loc[df['divida_moeda_estrangeira'].notna(), 'ano'].value_counts().sort_index().to_dict())
# Liquidez corrente do Economatica = AC / PC?
lc = (df['liquidez_corrente'] - df['ativo_circulante'] / df['passivo_circulante']).abs()
print('liquidez_corrente == AC/PC em', style.pct_br((lc < 1e-6).mean() / df['liquidez_corrente'].notna().mean(), 1), 'das linhas preenchidas')
# Dívida bruta = empréstimos CP + LP?
db = (df['divida_bruta'] - df['emprestimos_cp'] - df['emprestimos_lp']).abs()
print('divida_bruta == emprestimos_cp + emprestimos_lp em', style.pct_br((db < 1).sum() / db.notna().sum(), 1), 'dos casos')
# Valor de mercado existe sem balanço (cotação sem demonstração)?
print('linhas com valor_mercado mas sem ativo_total:', int((df['valor_mercado'].notna() & df['ativo_total'].isna()).sum()))

divida_moeda_estrangeira não nula por ano: {2010: 484, 2011: 1}
liquidez_corrente == AC/PC em 100,0% das linhas preenchidas
divida_bruta == emprestimos_cp + emprestimos_lp em 100,0% dos casos
linhas com valor_mercado mas sem ativo_total: 1090


## 5. Resumo da Fase 1 e decisões pendentes

**Base:** 3 exportações em formato longo (empresa × trimestre), 295 tickers, de 1T2010 a 2T2026 com dados (o 3T2026 e o 4T2026 ainda não foram divulgados), 14.827 empresa-trimestres com balanço. Sem duplicatas.

| Verificação | Resultado |
|---|---|
| a) Acumulado × 12 meses | `receita` e `desp_financeira` são **acumuladas no ano**; `ebit`, `ebitda`, `lucro_liquido_12m` e `caixa_operacional_12m` são de 12 meses. Usar só o 4T resolve. |
| Unidade do FCO | estava em R$; convertido para milhares (FCO/EBITDA mediano de 0,61). |
| b) Sinal da desp. financeira | despesa **positiva**. No 4T, 18 observações negativas e 9 iguais a zero (27 obs., 19 empresas) → cobertura NaN. |
| c) Resultado líquido | **não existe**: `ResLuc` é **Reservas de Lucro**. Renomeada para `reservas_lucro` e mantida para o Z''. Há 353 obs. negativas em 55 empresas, o que precisa ser entendido. |
| d) Passivo + PL = AT | sim (99,99%, diferenças de R$ 1 mil) → descartar `passivo_mais_pl`. |
| e) Duplicatas | nenhuma, nem por ticker nem por CNPJ. |
| f) Financeiras | **8 restantes** (B1003, B3SA3, BRBI11, CXSE3, HBTS5, ITSA4, QUAL3, WIZC3). Não excluídas, aguardando decisão. |
| g) Canceladas | **0 das 469** canceladas não financeiras estão na base; só 6 empresas param de reportar antes de 2025 (todas ativas na CVM). |
| Outros | `divida_moeda_estrangeira` só tem dados em **2010**; `liquidez_corrente` = AC/PC; `divida_bruta` = empréstimos CP + LP; 3 tickers sem nenhum balanço (B3SA3, CSUD3, G2DI33). |

**Decisões tomadas pela dupla (21/10/2026), aplicadas na Fase 2:**
1. **H5 (exposição cambial): abandonada**, porque a dívida em moeda estrangeira só existe para 2010.
2. **Financeiras: as 8 são excluídas**, incluindo as corretoras de seguro Qualicorp e Wiz.
3. **Setor:** a dupla vai exportar o **Subsetor Bovespa** do Economatica. Até lá a Fase 2 usa um agrupamento **provisório** do `SETOR_ATIV` da CVM em macrossetores (`cfg.SETOR_MACRO`).
4. **Anos antes do registro na CVM: mantidos**, com a flag `pre_registro`. Para análise de crédito o balanço vale mesmo antes do IPO. Divisão temporal: treino de 2010 a 2018 (cerca de 10 anos), validação/intervalo em 2019–2020 e teste de 2021 a 2023 (os mais recentes).
5. **Reservas de lucro negativas:** saldo líquido de reservas menos prejuízos acumulados, com o sinal mantido (seção 3c).
6. **Ligações ticker → CNPJ:** COMR3 (Comerc, adquirida pela Vibra), IGSN3 (Iguá, registrada mas nunca listada) e JBSS32 (BDR que substituiu JBSS3) estão confirmadas. EGGY3 ainda será conferida. LLBI3 está em recuperação judicial e teve o registro cancelado de ofício pela CVM, mas não foi encontrada no cadastro, então fica sem CNPJ.